# Estabilidad entre seeds v2: mejora relativa invariante a escala

Este notebook ejecuta el protocolo v2 congelado después del FAIL del gate de estabilidad original. Repite la misma condición corta de train/validation con seeds nuevas `5–9`; el dataset permanece fijo y sólo cambian la inicialización del modelo y el orden de minibatches.

La diferencia predefinida es una sola: la variabilidad entre seeds se evalúa sobre el ratio `validation seleccionada / validation inicial` de cada seed. La variabilidad de la loss absoluta se conserva como diagnóstico, pero no decide el gate. La configuración fue congelada en `configs/paper_seed_stability_scale_invariant.yaml` antes de ejecutar estas seeds. El split de test permanece reservado.

## Hipótesis y política predefinida

La loss JEPA cambia cuadráticamente si todo el espacio latente de una corrida cambia de escala. Para cada seed definimos `r_s = loss_validation_seleccionada / loss_validation_epoca_0`. Una reescala global de los embeddings de esa seed afecta numerador y denominador por el mismo factor y se cancela en `r_s`.

Un checkpoint sigue siendo elegible sólo si reduce validation al 50% o menos de su baseline, mantiene `validation/train ≤ 4`, conserva al menos 10% de dispersión, tiene rango efectivo ≥ 4 y presenta valores finitos. Entre checkpoints elegibles se elige la menor validation loss, con desempate a favor de la época más temprana.

El gate conjunto exige: checkpoint para las cinco seeds, peor `validation/baseline ≤ 0.5`, peor brecha ≤ 4, CV poblacional de los cinco ratios `r_s ≤ 0.25` y controles de representación en todas las seeds. El CV de las losses absolutas se mostrará al lado para comprobar cuánto cambia la lectura al quitar la escala, sin usarlo para aprobar o rechazar.

In [ ]:
# ruff: noqa: E402, E501
import json
import platform
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "No se encontró la raíz del repositorio."
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.paper_config import load_paper_scale_invariant_seed_stability_config
from koopman_jepa.paper_data import (
    PAPER_REGIME_NAMES,
    PaperRegimeDataset,
    generate_paper_master,
)
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    evaluate_scale_invariant_seed_stability_gate,
    run_paper_train_validation,
    select_validation_checkpoint,
    summarize_seed_checkpoint,
)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
config_path = ROOT / "configs" / "paper_seed_stability_scale_invariant.yaml"
config = load_paper_scale_invariant_seed_stability_config(config_path)
torch.use_deterministic_algorithms(True)

assert config.sweep.seeds == (5, 6, 7, 8, 9)
assert set(config.sweep.seeds).isdisjoint({0, 1, 2, 3, 4})
print(config_path.relative_to(ROOT))
print(json.dumps(asdict(config), indent=2))

## Dataset fijo y test cerrado

El `base_seed` del generador sigue fijo en 0. Las cinco corridas observan exactamente los mismos 576 masters de train y 144 de validation que el protocolo anterior; esto aísla variabilidad de optimización, no variabilidad de muestreo. Sólo se instancian train y validation.

In [ ]:
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
train_keys = {train_dataset.sample_key(index) for index in range(len(train_dataset))}
validation_keys = {
    validation_dataset.sample_key(index) for index in range(len(validation_dataset))
}

assert len(train_dataset) == 32 * len(PAPER_REGIME_NAMES) == 576
assert len(validation_dataset) == 8 * len(PAPER_REGIME_NAMES) == 144
assert train_keys.isdisjoint(validation_keys)
print(f"Train/validation: {len(train_dataset)}/{len(validation_dataset)}")
print(f"Dataset base seed: {config.data.base_seed}")
print("Intersección train/validation: 0 — PASS")
print(
    "Test reservado y no instanciado: "
    f"{config.data.test_per_regime * len(PAPER_REGIME_NAMES)} secuencias"
)

## Cinco trayectorias nuevas de optimización

Cada modelo se crea después de fijar su seed. El historial contiene la baseline sin entrenar de época 0 y diez épocas entrenadas. Un resultado sin checkpoint elegible se conserva como fallo; ningún límite se modifica durante el loop.

In [ ]:
histories = {}
selections = {}
summaries = []
run_times = {}

for seed in config.sweep.seeds:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    run_config = replace(config.train, seed=seed)
    model = PaperTemporalJEPA(config.model)

    started = time.perf_counter()
    history = run_paper_train_validation(
        model,
        train_dataset,
        validation_dataset,
        run_config,
    )
    run_times[seed] = time.perf_counter() - started
    selection = select_validation_checkpoint(history, config.checkpoint_gate)

    histories[seed] = history
    selections[seed] = selection
    if selection is not None:
        summaries.append(summarize_seed_checkpoint(seed, selection))
        print(
            f"seed={seed} checkpoint={selection.epoch} "
            f"val={selection.validation_loss:.6g} "
            f"val/base={selection.validation_loss_ratio:.3f} "
            f"gap={selection.validation_train_loss_ratio:.3f} "
            f"time={run_times[seed]:.2f}s"
        )
    else:
        print(f"seed={seed} checkpoint=NONE time={run_times[seed]:.2f}s")

gate = evaluate_scale_invariant_seed_stability_gate(
    summaries,
    config.sweep,
    config.stability_gate,
)
print(f"Tiempo total: {sum(run_times.values()):.2f} s")
print(json.dumps(asdict(gate), indent=2))

In [ ]:
print("seed  epoch  val_loss      val/base   val/train  std/base  val_rank")
for summary in summaries:
    print(
        f"{summary.seed:>4d}  {summary.checkpoint_epoch:>5d}  "
        f"{summary.validation_loss:>11.6g}  "
        f"{summary.validation_loss_ratio:>8.3f}   "
        f"{summary.validation_train_loss_ratio:>8.3f}  "
        f"{summary.validation_embedding_std_ratio:>8.3f}  "
        f"{summary.validation_effective_rank:>8.3f}"
    )

## Trayectorias y comparación de escalas

Los cuatro primeros paneles muestran todos los controles que intervienen en la selección. Los puntos con borde negro son los checkpoints elegidos. Los dos últimos comparan directamente las losses absolutas y las mejoras relativas; sólo el CV del panel relativo es criterio del gate v2.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)

for seed, history in histories.items():
    epochs = np.array([row.epoch for row in history])
    train_loss = np.array([row.train_loss for row in history])
    validation_loss = np.array([row.validation_loss for row in history])
    validation_std = np.array([row.validation_embedding_std for row in history])
    validation_rank = np.array([row.validation_effective_rank for row in history])
    gap = validation_loss / np.maximum(train_loss, 1e-12)
    line = axes[0, 0].plot(
        epochs,
        validation_loss / validation_loss[0],
        marker="o",
        markersize=3,
        label=f"seed {seed}",
    )[0]
    axes[0, 1].plot(epochs, gap, marker="o", markersize=3, color=line.get_color())
    axes[0, 2].plot(
        epochs,
        validation_std / validation_std[0],
        marker="o",
        markersize=3,
        color=line.get_color(),
    )
    axes[1, 0].plot(
        epochs, validation_rank, marker="o", markersize=3, color=line.get_color()
    )
    selection = selections[seed]
    if selection is not None:
        axes[0, 0].scatter(
            selection.epoch,
            selection.validation_loss_ratio,
            color=line.get_color(),
            edgecolor="black",
            s=70,
            zorder=5,
        )

axes[0, 0].axhline(
    config.checkpoint_gate.max_validation_loss_ratio,
    color="tab:red",
    linestyle="--",
    label="máximo elegible",
)
axes[0, 0].set(title="Validation frente a baseline", xlabel="Época", ylabel="Ratio de loss")
axes[0, 0].legend(ncol=2)

axes[0, 1].axhline(
    config.checkpoint_gate.max_validation_train_loss_ratio,
    color="tab:red",
    linestyle="--",
)
axes[0, 1].set(title="Brecha validation/train", xlabel="Época", ylabel="Ratio de loss")

axes[0, 2].axhline(
    config.checkpoint_gate.min_validation_embedding_std_ratio,
    color="tab:red",
    linestyle="--",
)
axes[0, 2].set(
    title="Dispersión de validation",
    xlabel="Época",
    ylabel="Ratio frente a época 0",
)

axes[1, 0].axhline(
    config.checkpoint_gate.min_validation_effective_rank,
    color="tab:red",
    linestyle="--",
)
axes[1, 0].axhline(config.model.latent_dim, color="0.5", linestyle=":")
axes[1, 0].set(
    title="Rango efectivo de validation",
    xlabel="Época",
    ylabel="exp(entropía espectral)",
)

if summaries:
    seed_values = np.array([summary.seed for summary in summaries])
    selected_losses = np.array([summary.validation_loss for summary in summaries])
    selected_ratios = np.array([summary.validation_loss_ratio for summary in summaries])
    axes[1, 1].bar(seed_values, selected_losses, color="tab:orange", alpha=0.8)
    axes[1, 2].bar(seed_values, selected_ratios, color="tab:purple", alpha=0.8)
axes[1, 1].set(
    title=f"Loss absoluta (CV diagnóstico={gate.absolute_validation_loss_coefficient_of_variation:.3f})",
    xlabel="Seed",
    ylabel="Validation loss",
)
axes[1, 2].axhline(
    config.stability_gate.max_worst_validation_loss_ratio,
    color="tab:red",
    linestyle="--",
    label="peor caso permitido",
)
axes[1, 2].set(
    title=f"Validation/baseline (CV gate={gate.validation_loss_ratio_coefficient_of_variation:.3f})",
    xlabel="Seed",
    ylabel="Ratio de loss",
)
axes[1, 2].legend()

plt.show()

In [ ]:
status = "PASS" if gate.passed else "FAIL"
criteria = {
    "checkpoints para todas las seeds": gate.all_checkpoints_selected,
    "métricas finitas": gate.all_finite,
    "mejora validation/baseline": gate.validation_loss_passed,
    "brecha validation/train": gate.generalization_gap_passed,
    "variabilidad relativa entre seeds": gate.variability_passed,
    "dispersión latente": gate.spread_passed,
    "rango efectivo": gate.rank_passed,
}
failed_criteria = [name for name, passed in criteria.items() if not passed]
failed_text = ", ".join(failed_criteria) if failed_criteria else "ninguno"

if summaries:
    worst_loss_summary = max(summaries, key=lambda row: row.validation_loss_ratio)
    worst_gap_summary = max(summaries, key=lambda row: row.validation_train_loss_ratio)
    worst_spread_summary = min(summaries, key=lambda row: row.validation_embedding_std_ratio)
    worst_rank_summary = min(summaries, key=lambda row: row.validation_effective_rank)
    selected_epoch_text = ", ".join(
        f"{summary.seed}:{summary.checkpoint_epoch}" for summary in summaries
    )
else:
    worst_loss_summary = worst_gap_summary = None
    worst_spread_summary = worst_rank_summary = None
    selected_epoch_text = "ninguno"

final_gaps = {
    seed: history[-1].validation_loss / max(history[-1].train_loss, 1e-12)
    for seed, history in histories.items()
}
ineligible_final_seeds = [
    seed
    for seed, gap in final_gaps.items()
    if gap > config.checkpoint_gate.max_validation_train_loss_ratio
]
ineligible_final_text = ", ".join(str(seed) for seed in ineligible_final_seeds) or "ninguna"

if gate.absolute_validation_loss_coefficient_of_variation > gate.validation_loss_ratio_coefficient_of_variation:
    scale_interpretation = (
        "Normalizar cada corrida por su propia baseline reduce la dispersión entre seeds. "
        "Eso es compatible con que parte de la variación absoluta provenga de la escala latente, "
        "aunque por sí solo no demuestra que ésa sea la única causa."
    )
else:
    scale_interpretation = (
        "Normalizar por la baseline no reduce la dispersión entre seeds. La variabilidad observada "
        "no puede atribuirse simplemente a una escala latente global."
    )

decision = (
    "El gate v2 pasa. El próximo paso permitido es congelar el protocolo de evaluación test; "
    "test no se abre dentro de este notebook."
    if gate.passed
    else "El gate v2 falla. Test permanece cerrado y corresponde diagnosticar los criterios fallidos "
    "sin relajar retrospectivamente sus umbrales."
)

display(Markdown(f"""## Resultado e interpretación

- **Gate global: {status}.**
- **Criterios fallidos:** {failed_text}.
- **Checkpoints:** {'PASS' if gate.all_checkpoints_selected else 'FAIL'}; épocas por seed = `{selected_epoch_text}`.
- **Variabilidad relativa:** CV de `validation/baseline` = `{gate.validation_loss_ratio_coefficient_of_variation:.3f}` frente al máximo `{config.stability_gate.max_validation_loss_ratio_coefficient_of_variation:.2f}`. Éste es el criterio v2.
- **Variabilidad absoluta:** CV de validation loss = `{gate.absolute_validation_loss_coefficient_of_variation:.3f}`. Es diagnóstico y no interviene en PASS/FAIL.
- **Peor mejora:** validation/baseline = `{gate.worst_validation_loss_ratio:.3f}` frente al máximo `{config.stability_gate.max_worst_validation_loss_ratio:.2f}`.
- **Peor brecha:** validation/train = `{gate.worst_validation_train_loss_ratio:.3f}` frente al máximo `{config.stability_gate.max_worst_validation_train_loss_ratio:.1f}`.
- **Peor dispersión:** retención `{gate.worst_validation_embedding_std_ratio:.1%}` frente al mínimo `{config.stability_gate.min_worst_validation_embedding_std_ratio:.1%}`.
- **Peor rango efectivo:** `{gate.worst_validation_effective_rank:.3f}` frente al mínimo `{config.stability_gate.min_worst_validation_effective_rank:.1f}`.

### Lectura de los gráficos

Los puntos con borde negro del primer panel son los checkpoints elegidos. Las trayectorias de brecha, dispersión y rango permiten verificar que una loss baja no se acepte a costa de overfitting o colapso. Las épocas finales que exceden la brecha 4 corresponden a seeds `{ineligible_final_text}` y, si existe una época anterior elegible, la política debe seleccionarla.

Los peores casos seleccionados pertenecen a seed `{worst_loss_summary.seed if worst_loss_summary else 'N/A'}` en mejora, seed `{worst_gap_summary.seed if worst_gap_summary else 'N/A'}` en brecha, seed `{worst_spread_summary.seed if worst_spread_summary else 'N/A'}` en dispersión y seed `{worst_rank_summary.seed if worst_rank_summary else 'N/A'}` en rango.

Los dos paneles de barras contestan la pregunta central de este protocolo. {scale_interpretation}

### Decisión

{decision} Esta prueba mantiene fijo el dataset: **no mide** variabilidad entre muestras y todavía **no reproduce** la pureza de clustering ni los diagnósticos del operador del paper. El split de test no fue instanciado ni consultado.
"""))